# Contribution workflow: reproduce before proposing a change

This is a local, already-resolved Qiskit case—not a new contribution opportunity or a Rust course. You need Python/basic circuits; no prior completion of Labs 01–02 is required. Work through the written prompts yourself. Running every cell does **not** complete the human review.

[Workflow guide](../docs/contribution-workflow.md) · [Setup help](../docs/setup.md) · [Case commands and source map](../docs/contribution-case.md)

Original AI-assisted teaching material. Upstream case: [issue #17113](https://github.com/Qiskit/qiskit/issues/17113), reported by rishabhhP, and [fix #17117](https://github.com/Qiskit/qiskit/pull/17117), authored by davidfcohen. Attribution does not imply involvement or endorsement. Delay reading the fix until after your prediction/reproduction if you want to practice diagnosis.

**Material version: contribution-workflow-r1-2026-11-01.**

**Which layer are we studying?** In this case, `QDrift.expand` builds a rotation list in Python. Calling it alone does not execute the Rust synthesis path. Later, `ProductFormula.synthesize` calls `expand` and, on its default path, passes the result to a Rust-backed routine. You will locate those separate steps; you do not need to learn Rust or edit Rust code for this exercise.

The notebook kernel uses the fixed release, 2.6.0. The observation cell below demonstrates the fixed zero-operator behavior only; step 3 runs the historical comparison in separate environments.

## 1. Start with the contribution rules

Open [Qiskit's current CONTRIBUTING.md](https://github.com/Qiskit/qiskit/blob/main/CONTRIBUTING.md) and its linked policies. Write your answers below:

- Where would you check whether a proposed task is already solved or being worked on?
- What should be discussed before a PR? Which tests, formatting, and release-note requirements might apply?
- What do the current AI policy and contributor agreement require? Reading the agreement here does not sign it.
- Why should this historical fix **not** be submitted again?

**Your answers:** _Write here. Record the date and policy revision you consulted._

In [ ]:
import sys

import qiskit

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Expected teaching release: 2.6.0 (fixed). Historical comparison: 2.5.2.")

## 2. Predict the behavior

Time evolution under an operator H is U(t) = exp(-i t H). If H is zero, or time is zero, U is identity. QDrift expresses an approximate evolution as sampled Pauli rotations. In these zero cases, an empty rotation list is appropriate; it does not mean the circuit has lost its qubits.

Predict what `QDrift.expand` should return for (coefficient, time) = (0, 2), (1, 0), and (1, 1). Should a valid zero case emit divide-by-zero warnings? Explain before running the next cell.

**Your prediction and reasoning:** _Write here._

In [ ]:
import warnings

from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import QDrift

# Observe the installed release; no monkeypatch or simulated old failure.
zero_evolution = PauliEvolutionGate(SparsePauliOp("Z", coeffs=[0.0]), time=2.0)
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter("always")
    try:
        expansion = QDrift(seed=7).expand(zero_evolution)
        print("Zero-operator expansion:", expansion)
    except (ValueError, FloatingPointError, ZeroDivisionError) as exc:
        print("Observed exception:", type(exc).__name__, str(exc))
print("Warnings:", len(observed_warnings))
for warning in observed_warnings:
    print(warning.category.__name__, str(warning.message))

### Pause: save your initial test idea

Before opening the case guide, bundled tests, or repair diff, write a small test idea in your own words: inputs, expected result, and what would count as failure. Save this initial sketch before refining it later. If you have already seen any reference, say what you saw. You can still practice, but the result is reference-informed rather than an unaided design.

**Initial sketch and prior exposure:** _Write here._

## 3. Compare an actual old and fixed release

Use the two isolated-environment commands in the [case guide](../docs/contribution-case.md#run-the-historical-comparison). They run the same local regression file against 2.5.2 and 2.6.0 without changing this notebook's environment. Downloads may need internet. A nonzero test exit in 2.5.2 is expected; a dependency-install failure is **not** a reproduced bug.

**Record:** package/Python versions, commands, results, warnings, and help used. If you cannot run both, say which run is missing. A fixed-wheel demonstration is not a source build or a test of your own patch.

**Your observation record:** _Write here; do not paste personal paths or tokens._

## 4. Locate the relevant implementation and tests

Follow the pinned source map in the [case guide](../docs/contribution-case.md#source-navigation). Find the zero-input handling in Python `QDrift.expand`, its nearby upstream tests, and the subsequent Python-to-Rust call in `ProductFormula.synthesize`.

**Your source map:** public method → Python implementation → relevant tests → Rust binding used later by synthesis. Name the language of each. Explain why calling `expand` alone does not exercise that later Rust call, and why this particular repair belongs in Python.

**Your map and explanation:** _Write here. Reading Rust paths is not Rust implementation proficiency._

## 5. Refine your regression; avoid a weak test

Revisit the initial sketch you saved before step 3. Keep it alongside your revised test and describe what changed after reading the case guide, source, or tests. Record exactly which references and help you used; changing a Pauli or seed alone does not establish independent test design.

- What known result supplies the assertion, independently of the implementation?
- Is asserting an empty list sufficient when time is zero? Inspect the warning behavior of the old release.
- What nonzero case would guard against an implementation that always returns an empty list?
- What does one deterministic single-term example **not** establish about randomized multi-term synthesis?

Write your own trusted test in a separate local file and run it in both environments using the case guide. Do not change the bundled reference just to make an old release pass.

**Your regression, rationale, references, and limits:** _Write here._

In [ ]:
# A one-term control has no choice between different Pauli terms.
control = PauliEvolutionGate(SparsePauliOp("Z", coeffs=[1.0]), time=1.0)
print("Nonzero control:", QDrift(seed=7).expand(control))
print("This observation does not prove general QDrift correctness.")

## 6. Prepare a human-review packet

Use the [practice-review template](../docs/practice-review.md). Include a small reproducer, actual results, source/test map, proposed explanation of the historical repair, test rationale, compatibility questions, and help used. A human reviewer will choose a follow-up scenario and check whether you have seen it before. Record your first answer and any later hints separately. A previously seen or prompted answer is useful practice, not unaided transfer evidence.

You may stop with a documented prerequisite or setup blocker. No upstream submission, public post, agreement signing, or message is part of this exercise. Future real tasks must be genuinely useful and comply with then-current project policy.

**Reviewer's question, prior exposure, first answer, prompts, and next preparation needed:** _Leave pending until an actual review occurs._

In [ ]:
print("Walkthrough execution complete.")
print("Human review: pending")
print("No automatic contribution-readiness grade is assigned.")